# 15 — CONTEXTUAL_SEMANTIC_STACK_V1

Final architecture experiment on the already-consumed 101-pair DEV corpus.

**Candidate predictor exclusions:** Production Control score, Control breakdown, age, axis, chronology, nuisance, event metadata.

**Stopping rule:** PASS → freeze and collect fresh confirmation DEV. FAIL → permanently stop model/architecture search on these 101 pairs.

Control ties are scored **0.5**, matching Notebook 12.

In [1]:

from pathlib import Path
from datetime import datetime
from collections import Counter
import hashlib, inspect, json, pickle, sys, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings('ignore')

NOTEBOOK_VERSION='SAJU_ML_V4_CONTEXTUAL_SEMANTIC_STACK_V1_20260817'
SEED=20260817; OUTER_FOLDS=5; OUTER_REPEATS=10; N_BOOTSTRAP=20000; MODEL_C=0.3
MIN_OVERALL=.56; MIN_DELTA_CONTROL=.01; MIN_BOOT_P=.70; MIN_P10=.50; MIN_AXIS=.48
EXPECTED_ENGINE_SHA='d39e0c4d777ae3a19394c9175319a4f8a6a709c2b4d34400965b618ffbfdad1e'

def root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p]+list(p.parents):
        if (c/'saju_engine.py').exists(): return c
    raise FileNotFoundError('Run inside Chartpalja repo')

def sha(p):
    h=hashlib.sha256();
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()

ROOT=root()
PAIR_PATH=ROOT/'research/ml/artifacts/v4_unified_dev_wave2/V4_UNIFIED_DEV_COMBINED_FROZEN_PAIRS.csv'
W1=ROOT/'research/ml/artifacts/v4_unified_dev_roster/V4_UNIFIED_DEV_SUBJECT_ROSTER_100.csv'
W2=ROOT/'research/ml/artifacts/v4_unified_dev_wave2_roster/V4_UNIFIED_DEV_WAVE2_SUBJECT_ROSTER_44.csv'
PRIOR=ROOT/'research/ml/artifacts/v4_unified_dev_3arch_tournament/V4_UNIFIED_DEV_3ARCH_TOURNAMENT_DECISION.json'
FAIL=ROOT/'research/ml/artifacts/v4_control_failure_diagnostics/V4_CONTROL_FAILURE_DIAGNOSTIC_DECISION.json'
REJECT=ROOT/'research/ml/artifacts/v4_structured_factor_stack_v1/V4_STRUCTURED_FACTOR_STACK_V1_DECISION.json'
SPEC=ROOT/'research/ml_corpus/v4_contextual_semantic_stack/V4_CONTEXTUAL_SEMANTIC_STACK_V1_SPEC.json'
ART=ROOT/'research/ml/artifacts/v4_contextual_semantic_stack_v1'; ART.mkdir(parents=True,exist_ok=True)
for p in [PAIR_PATH,W1,W2,PRIOR,FAIL,REJECT,SPEC]:
    if not p.exists(): raise FileNotFoundError(p)

pairs=pd.read_csv(PAIR_PATH); w1=pd.read_csv(W1); w2=pd.read_csv(W2)
prior=json.load(open(PRIOR,encoding='utf-8')); fail=json.load(open(FAIL,encoding='utf-8')); rej=json.load(open(REJECT,encoding='utf-8')); spec=json.load(open(SPEC,encoding='utf-8'))
assert prior['status']=='V4_UNIFIED_DEV_NO_ARCHITECTURE_SURVIVES_DISCOVERY_GATE'
assert fail['status']=='V4_CONTROL_FAILURE_DIAGNOSTICS_COMPLETE_REDESIGN_BEFORE_FRESH_CONFIRM_DEV'
assert rej['status']=='V4_STRUCTURED_FACTOR_STACK_V1_REJECTED_NO_FRESH_CONFIRM'
assert len(pairs)==101 and pairs.subject_id.nunique()==101 and pairs.pair_id.nunique()==101
assert pairs.groupby('axis').size().to_dict()=={'COMPETITIVE':31,'PROJECT':25,'STATUS':45}
assert all(v is False for v in prior['holdout_integrity'].values())
assert all(v is False for v in fail['holdout_integrity'].values())
assert all(v is False for v in rej['holdout_integrity'].values())
engine_sha=sha(ROOT/'saju_engine.py')
if engine_sha!=EXPECTED_ENGINE_SHA: raise RuntimeError('STOP engine SHA changed: '+engine_sha)

sys.path[:0]=[str(ROOT),str(ROOT/'test'),str(ROOT/'test/experiments')]
import saju_engine as se, sajupy
from experiments import experiment_v2_dy_orthodox as O
ORTHO=ROOT/'test/experiments/experiment_v2_dy_orthodox.py'
print('Preflight PASS',engine_sha,sha(ORTHO))


Preflight PASS d39e0c4d777ae3a19394c9175319a4f8a6a709c2b4d34400965b618ffbfdad1e 8db818ad59ed5109fdb658fe778deeffc883f55b6f0af924d9786487bbbd3390


In [2]:

# Helper-source leakage audit.
helpers={'_natal_context':O._natal_context,'_regime_evidence':O._regime_evidence,'_contextual_year':O._contextual_year,'_ten_god_role':O._ten_god_role,'_classify_relation_valence':O._classify_relation_valence,'_elem_sign':O._elem_sign}
forbidden=['candle.close','scores.종합','종합운점수','_composite_score','SCORE_BIAS','V2_DY_B','G_CLEAN','A_G']
a=[]
for name,fn in helpers.items():
    src=inspect.getsource(fn); hits=[x for x in forbidden if x in src]
    a.append({'helper':name,'signature':str(inspect.signature(fn)),'forbidden_hits':'|'.join(hits),'pass':not hits})
a=pd.DataFrame(a); display(a); assert a['pass'].all(); a.to_csv(ART/'V4_CONTEXTUAL_HELPER_LEAKAGE_AUDIT.csv',index=False)


,helper,signature,forbidden_hits,pass
0,_natal_context,"(pack: 'dict') -> 'Dict[str, Any]'",,True
1,_regime_evidence,"(nc: 'dict', feat: 'dict', stem: 'str', branch...",,True
2,_contextual_year,"(nc: 'dict', meta: 'dict', d_stem: 'str', d_br...",,True
3,_ten_god_role,"(tg: 'str', nc: 'dict') -> 'Tuple[float, str, ...",,True
4,_classify_relation_valence,"(pairs: 'Sequence[str]', target_idx, nc: 'dict...",,True
5,_elem_sign,"(elem: 'str', nc: 'dict') -> 'float'",,True


In [3]:

# Same 101 birth lineage and longitude-correction policy as Notebook 12.
all_roster=pd.concat([w1,w2],ignore_index=True,sort=False); ids=set(pairs.subject_id)
subject_map={}
for _,r in all_roster.iterrows():
    if r.subject_id not in ids: continue
    subject_map[r.subject_id]={'subject_id':r.subject_id,'name':r['name'],'gender':r['gender'],'birth':{'calendar':'solar','date':str(r['birth_date']),'time':str(r['birth_time']),'utc_offset':str(r['utc_offset']),'longitude':float(r['longitude']),'latitude':float(r['latitude']),'place':str(r['birth_place'])}}
assert len(subject_map)==101

def parse_off(raw):
    if isinstance(raw,(int,float)): return float(raw)
    s=str(raw).strip(); sign=-1. if s.startswith('-') else 1.; s=s[1:] if s[:1] in '+-' else s; hh,mm=s.split(':'); return sign*(int(hh)+int(mm)/60.)

def corr_birth(b):
    y,m,d=map(int,b['date'].split('-')); hh,mi=map(int,b['time'].split(':')[:2]); calc=sajupy.get_saju_calculator(); corr=float(calc._calculate_solar_time_correction(float(b['longitude']),parse_off(b['utc_offset']))); h2,m2,dc=calc._adjust_time_for_solar(hh,mi,corr); y2,mo2,d2=calc._adjust_date_for_solar(y,m,d,dc); return {'y':int(y2),'m':int(mo2),'d':int(d2),'h':int(h2),'min':int(m2),'corr':corr}

def gender(x):
    s=str(x).lower(); return 'female' if s.startswith('female') or s in {'f','woman','여','여성'} else 'male'

def build_pack(s):
    c=corr_birth(s['birth']); inp=se.BirthInput(year=c['y'],month=c['m'],day=c['d'],hour=c['h'],minute=c['min'],gender=gender(s['gender']),calendar='solar',is_leap_month=False,use_solar_time=False,utc_offset=9); r=se.compute_all(inp); dw=se.build_daewoon_detail(r); meta={int(x['year']):x for x in r['chart_data']['연도별_타임라인']}; return {'name':s['name'],'subject_id':s['subject_id'],'r':r,'dw':dw,'meta':meta,'corrected_birth':c}

packs={}
for i,sid in enumerate(sorted(subject_map)):
    packs[sid]=build_pack(subject_map[sid])
    if (i+1)%10==0 or i+1==101: print('packs',i+1,'/101')


[SAJU_DEBUG] original_input: 1969-10-13 16:32
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1969-10-13
[SAJU_DEBUG] final_datetime(KST): 1969-10-13T16:32:00+09:00
[SAJU_DEBUG] pillars: 연=己酉 월=甲戌 일=辛酉 시=丙申
[SAJU_DEBUG] original_input: 1940-02-03 00:32
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1940-02-03
[SAJU_DEBUG] final_datetime(KST): 1940-02-03T00:32:00+09:00
[SAJU_DEBUG] pillars: 연=己卯 월=丁丑 일=丙子 시=戊子
[SAJU_DEBUG] original_input: 1958-06-15 20:45
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1958-06-15
[SAJU_DEBUG] final_datetime(KST): 1958-06-15T20:45:00+09:00
[SAJU_DEBUG] pillars: 연=戊戌 월=戊午 일=癸亥 시=壬戌
[SAJU_DEBUG] original_input: 1946-07-15 17:15
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female


packs 10 /101


[SAJU_DEBUG] original_input: 1924-10-15 16:58
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1924-10-15
[SAJU_DEBUG] final_datetime(KST): 1924-10-15T16:58:00+09:00
[SAJU_DEBUG] pillars: 연=甲子 월=甲戌 일=丁卯 시=戊申
[SAJU_DEBUG] original_input: 1943-03-19 05:05
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1943-03-19
[SAJU_DEBUG] final_datetime(KST): 1943-03-19T05:05:00+09:00
[SAJU_DEBUG] 반시보정: 辛卯→庚寅 (05:05)
[SAJU_DEBUG] pillars: 연=癸未 월=乙卯 일=丙子 시=庚寅
[SAJU_DEBUG] original_input: 1950-03-04 10:01
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1950-03-04
[SAJU_DEBUG] final_datetime(KST): 1950-03-04T10:01:00+09:00
[SAJU_DEBUG] pillars: 연=庚寅 월=戊寅 일=戊戌 시=丁巳
[SAJU_DEBUG] original_input: 1936-01-06 01:15
[SAJU_DEBUG] calendar=solar, is_l

packs 20 /101


[SAJU_DEBUG] original_input: 1918-02-05 11:54
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1918-02-05
[SAJU_DEBUG] final_datetime(KST): 1918-02-05T11:54:00+09:00
[SAJU_DEBUG] pillars: 연=戊午 월=甲寅 일=癸未 시=戊午
[SAJU_DEBUG] original_input: 1955-02-24 19:05
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1955-02-24
[SAJU_DEBUG] final_datetime(KST): 1955-02-24T19:05:00+09:00
[SAJU_DEBUG] 반시보정: 戊戌→丁酉 (19:05)
[SAJU_DEBUG] pillars: 연=乙未 월=戊寅 일=丙辰 시=丁酉
[SAJU_DEBUG] original_input: 1937-09-02 12:18
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1937-09-02
[SAJU_DEBUG] final_datetime(KST): 1937-09-02T12:18:00+09:00
[SAJU_DEBUG] pillars: 연=丁丑 월=戊申 일=壬辰 시=丙午
[SAJU_DEBUG] original_input: 1936-04-19 10:44
[SAJU_DEBUG] calendar=solar, is_l

...

[SAJU_DEBUG] original_input: 1951-02-20 08:22
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1951-02-20
[SAJU_DEBUG] final_datetime(KST): 1951-02-20T08:22:00+09:00
[SAJU_DEBUG] pillars: 연=辛卯 월=庚寅 일=辛卯 시=壬辰
[SAJU_DEBUG] original_input: 1917-02-27 01:27
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1917-02-27
[SAJU_DEBUG] final_datetime(KST): 1917-02-27T01:27:00+09:00
[SAJU_DEBUG] 반시보정: 丁丑→丙子 (01:27)
[SAJU_DEBUG] pillars: 연=丁巳 월=壬寅 일=庚子 시=丙子
[SAJU_DEBUG] original_input: 1939-08-09 16:57
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1939-08-09
[SAJU_DEBUG] final_datetime(KST): 1939-08-09T16:57:00+09:00
[SAJU_DEBUG] pillars: 연=己卯 월=壬申 일=戊寅 시=庚申
[SAJU_DEBUG] original_input: 1955-01-28 21:09
[SAJU_DEBUG] calendar=solar, is_l

packs 90 /101


[SAJU_DEBUG] original_input: 1936-08-29 18:05
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1936-08-29
[SAJU_DEBUG] final_datetime(KST): 1936-08-29T18:05:00+09:00
[SAJU_DEBUG] pillars: 연=丙子 월=丙申 일=癸未 시=辛酉
[SAJU_DEBUG] original_input: 1924-04-12 06:11
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1924-04-12
[SAJU_DEBUG] final_datetime(KST): 1924-04-12T06:11:00+09:00
[SAJU_DEBUG] pillars: 연=甲子 월=戊辰 일=辛酉 시=辛卯
[SAJU_DEBUG] original_input: 1932-11-29 12:09
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1932-11-29
[SAJU_DEBUG] final_datetime(KST): 1932-11-29T12:09:00+09:00
[SAJU_DEBUG] pillars: 연=壬申 월=辛亥 일=甲午 시=庚午
[SAJU_DEBUG] original_input: 1925-11-15 14:21
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJ

packs 100 /101
packs 101 /101


[SAJU_DEBUG] original_input: 1938-04-07 12:24
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1938-04-07
[SAJU_DEBUG] final_datetime(KST): 1938-04-07T12:24:00+09:00
[SAJU_DEBUG] pillars: 연=戊寅 월=丙辰 일=己巳 시=庚午


In [4]:

# Clean contextual feature extraction. B._block_feats is intentionally never called.
STEMS=set('甲乙丙丁戊己庚辛壬癸'); BR=set('子丑寅卯辰巳午未申酉戌亥')
REG=['elemental_environment_shift','yong_xiang_support_shift','structural_activation_shift','geju_state_change','special_structure_change','tiaohou_change','rooting_exposure_change','major_relation_change','key_pillar_change','stem_branch_convergence']
REL_ROLES=['COMPETING_HAP_CLASH','AMBIGUOUS','SUPPORTS_USEFUL_OR_BINDS','SUPPORTS_HARMFUL','CLASHES_USEFUL_STRUCTURE','CLASHES_HARMFUL_STRUCTURE','STRUCTURAL_MOVEMENT']
TG_ROLES=['SUPPORTS_USEFUL_STRUCTURE','CONTROLS_EXCESS','OVERBURDENS_WEAK_STRUCTURE','DRAINS_EXCESS','DRAINS_WEAK','SUPPLIES_NEEDED_RESOURCE','FEEDS_HARMFUL_EXCESS','OVERBURDENS_OR_COVERS','AMBIGUOUS','NEUTRAL']

def pc(v):
    if isinstance(v,(list,tuple)) and len(v)>=2 and str(v[0]) in STEMS and str(v[1]) in BR: return str(v[0]),str(v[1])
    s=str(v); return next((x for x in s if x in STEMS),None),next((x for x in s if x in BR),None)

def drow(pack,m):
    ts,tb=pc(m.get('대운_pillar') or m.get('대운') or '')
    for r in pack.get('dw') or []:
        rs,rb=pc(r.get('daewoon_pillar') or '')
        if ts and tb and rs==ts and rb==tb: return r
    y=int(m['year'])
    for r in pack.get('dw') or []:
        if int(r['start_year'])<=y<int(r['end_year']): return r
    raise RuntimeError(('no D row',pack['name'],y))

def relagg(rels,nc):
    val=inten=0.; roles=[]
    for r in rels if isinstance(rels,list) else []:
        if not isinstance(r,dict): continue
        pp=r.get('relations') or r.get('patterns') or r.get('pairs') or []; pp=[pp] if isinstance(pp,str) else pp
        v,it,rr=O._classify_relation_valence(pp,r.get('pillar_idx'),nc); val+=float(v); inten+=float(it); roles+=rr
    return val,inten,roles

def feat_year(pack,year):
    m=pack['meta'][int(year)]; nc=O._natal_context(pack); dr=drow(pack,m); ds,db=pc(dr.get('daewoon_pillar') or ''); ds=ds or dr.get('stem'); db=db or dr.get('branch'); rels=dr.get('관계_with_원국') or []
    ssign=float(O._elem_sign(se.STEM_ELEMENT.get(ds,''),nc)); bsign=float(O._elem_sign(se.BRANCH_ELEMENT_MAIN.get(db,''),nc)); fm=.5*(ssign+bsign)
    clean={'fav_minus_unfav':fm,'struct_activ':0.,'struct_disrupt':0.,'struct_excess':0.}
    ev=O._regime_evidence(nc,clean,ds,db,rels); cy=O._contextual_year(nc,m,ds,db,float(ev['direction_score']))
    out={}
    for k in REG:
        state=str(ev['dims'].get(k,'inactive')); out['regime__signed__'+k]=float(ev['signed'].get(k,0.)); out['regime__active__'+k]=float(state!='inactive'); out['regime__ambiguous__'+k]=float(state in ('ambiguous','mixed'))
    for k,v in {'direction_score':ev['direction_score'],'contradiction_count':ev['contradiction_count'],'confidence':ev['confidence'],'n_active':ev['n_active'],'n_agree':ev['n_agree'],'relation_intensity':ev['event_intensity'],'season_efficacy':ev['season_efficacy'],'stem_rooted':float(bool(ev['stem_rooted'])),'fav_minus_unfav_clean':fm}.items(): out['regime__'+k]=float(v)
    rc=Counter(ev.get('relation_roles') or [])
    for r in REL_ROLES: out['regime_relrole__'+r]=float(rc[r])
    for k in ['contextual_trigger','dy_context','event_intensity','valence','reinforce','fuyin_natal_year','fuyin_natal_dae','binglin_dy','fanyin']: out['yearctx__'+k]=float(cy.get(k,0.) or 0.)
    for label,tg in [('stem',m.get('세운_십성_천간') or ''),('branch',m.get('세운_십성_지지') or '')]:
        eff,role,_=O._ten_god_role(tg,nc) if tg else (0.,'NEUTRAL',''); out[f'tgctx__{label}_effect']=float(eff)
        for rv in TG_ROLES: out[f'tgctx__{label}_role__{rv}']=float(role==rv)
    yv,yi,yr=relagg(m.get('세운_관계_with_원국') or [],nc); out['year_natal_rel__valence']=float(yv); out['year_natal_rel__intensity']=float(yi); yc=Counter(yr)
    for r in REL_ROLES: out['year_natal_rel__role__'+r]=float(yc[r])
    ys=m.get('세운_stem') or (m.get('세운_pillar') or '  ')[0]; yb=m.get('세운_branch') or (m.get('세운_pillar') or '  ')[1]; out['year_elem__stem_sign']=float(O._elem_sign(se.STEM_ELEMENT.get(ys,''),nc)); out['year_elem__branch_sign']=float(O._elem_sign(se.BRANCH_ELEMENT_MAIN.get(yb,''),nc))
    ctrl=(m.get('candle') or {}).get('close'); ctrl=ctrl if ctrl is not None else (m.get('scores') or {}).get('종합'); return out,float(ctrl)

rows=[]
for i,r in pairs.iterrows():
    for pol,y in [('positive',int(r.positive_year)),('negative',int(r.negative_year))]:
        f,c=feat_year(packs[r.subject_id],y); rows.append({'pair_id':r.pair_id,'subject_id':r.subject_id,'axis':r.axis,'positive_earlier_calc':bool(int(r.positive_year)<int(r.negative_year)),'polarity':pol,'year':y,'benchmark__control_score':c,**f})
    if (i+1)%10==0 or i+1==101: print('feature rows',2*(i+1),'/202')
yf=pd.DataFrame(rows); assert len(yf)==202; yf.to_csv(ART/'V4_CONTEXTUAL_SEMANTIC_YEAR_FEATURES.csv',index=False)


feature rows 20 /202
feature rows 40 /202
feature rows 60 /202
feature rows 80 /202
feature rows 100 /202
feature rows 120 /202
feature rows 140 /202
feature rows 160 /202
feature rows 180 /202
feature rows 200 /202
feature rows 202 /202


In [6]:
# Pair differences. No label-based feature selection; zero-variance only.

meta = {
    'benchmark__control_score',
    'pair_id',
    'subject_id',
    'axis',
    'positive_earlier_calc',
    'polarity',
    'year'
}

ycols = sorted([c for c in yf.columns if c not in meta])

# Leakage guard:
# Do NOT reject legitimate semantic labels such as CONTROLS_EXCESS.
# Only reject explicit benchmark / nuisance / chronology / metadata predictors.
def is_forbidden_candidate_col(c):
    x = str(c).lower()

    forbidden_exact = {
        'benchmark__control_score',
        'control_score',
        'candle.close',
        'scores.종합',
        '종합운점수',
        'axis',
        'positive_earlier',
        'positive_earlier_calc',
        'chronology',
        'age',
    }

    forbidden_substrings = [
        'benchmark__',
        'candle__',
        'nuisance__',
        'chronology__',
        'event_description',
        'source_domain',
        'source_url',
        'collection_wave',
        'pair_origin',
    ]

    if x in forbidden_exact:
        return True

    if any(tok in x for tok in forbidden_substrings):
        return True

    return False


bad = [c for c in ycols if is_forbidden_candidate_col(c)]
assert not bad, bad


pos = yf[yf.polarity == 'positive'].set_index('pair_id')
neg = yf[yf.polarity == 'negative'].set_index('pair_id')

pr = []

for _, r in pairs.iterrows():
    x = {
        'pair_id': r.pair_id,
        'subject_id': r.subject_id,
        'axis': r.axis,
        'positive_earlier_calc': bool(
            int(r.positive_year) < int(r.negative_year)
        ),
        'control_diff': float(
            pos.loc[r.pair_id, 'benchmark__control_score']
            - neg.loc[r.pair_id, 'benchmark__control_score']
        )
    }

    for c in ycols:
        x['diff__' + c] = float(
            pos.loc[r.pair_id, c]
            - neg.loc[r.pair_id, c]
        )

    pr.append(x)


pdff = pd.DataFrame(pr)

dcols = [
    c for c in pdff.columns
    if c.startswith('diff__')
]

use = [
    c for c in dcols
    if float(pdff[c].var()) > 0
]

zero = sorted(set(dcols) - set(use))

assert use


schema = {
    'architecture': 'CONTEXTUAL_SEMANTIC_STACK_V1',
    'n_year_features': len(ycols),
    'n_pair_features_after_zero_var': len(use),
    'candidate_pair_features': use,
    'zero_variance_removed': zero,
    'cleanliness': (
        'B._block_feats not called; '
        'struct_* score-connected inputs forced zero; '
        'Control only benchmark'
    )
}

json.dump(
    schema,
    open(
        ART / 'V4_CONTEXTUAL_SEMANTIC_FEATURE_SCHEMA.json',
        'w',
        encoding='utf-8'
    ),
    ensure_ascii=False,
    indent=2
)

pdff.to_csv(
    ART / 'V4_CONTEXTUAL_SEMANTIC_PAIR_DIFF.csv',
    index=False
)

print('features', len(use), 'zero-var', len(zero))
print('leakage guard PASS')

features 74 zero-var 14
leakage guard PASS


In [7]:

# Fixed repeated subject CV and official Control tie rule.
def fit(frame):
    X=frame[use].to_numpy(float); Xa=np.vstack([X,-X]); ya=np.r_[np.ones(len(X),int),np.zeros(len(X),int)]; m=Pipeline([('scale',StandardScaler()),('model',LogisticRegression(penalty='l2',C=MODEL_C,solver='liblinear',max_iter=5000,random_state=SEED))]); m.fit(Xa,ya); return m

def correct(x,eps=1e-12):
    x=np.asarray(x,float); return np.where(x>eps,1.,np.where(x<-eps,0.,.5))

def splits(frame,n,seed):
    z=frame[['subject_id','axis','positive_earlier_calc']].copy(); z['stratum']=z.axis.astype(str)+'__'+z.positive_earlier_calc.astype(int).astype(str); cnt=z.stratum.value_counts(); assert int(cnt.min())>=n; sk=StratifiedKFold(n_splits=n,shuffle=True,random_state=int(seed)); X=np.zeros((len(z),1)); y=z.stratum.to_numpy(); return [(z.iloc[tr].subject_id.tolist(),z.iloc[te].subject_id.tolist()) for tr,te in sk.split(X,y)]

o=[]
for rep in range(OUTER_REPEATS):
    for fold,(trids,teids) in enumerate(splits(pdff,OUTER_FOLDS,SEED+1000*rep)):
        tr=pdff[pdff.subject_id.isin(trids)]; te=pdff[pdff.subject_id.isin(teids)]; m=fit(tr); mar=m.decision_function(te[use].to_numpy(float)); cc=correct(mar); bc=correct(te.control_diff.to_numpy(float))
        for i,(_,r) in enumerate(te.iterrows()): o.append({'repeat':rep,'fold':fold,'pair_id':r.pair_id,'subject_id':r.subject_id,'axis':r.axis,'positive_earlier_calc':bool(r.positive_earlier_calc),'candidate_margin':float(mar[i]),'candidate_correct':float(cc[i]),'control_diff':float(r.control_diff),'control_correct':float(bc[i])})
    print('repeat',rep+1,'/10')
oof=pd.DataFrame(o); oof.to_csv(ART/'V4_CONTEXTUAL_SEMANTIC_OOF.csv',index=False)
rep=oof.groupby('repeat').agg(candidate=('candidate_correct','mean'),control=('control_correct','mean')).reset_index(); rep['delta']=rep.candidate-rep.control
cm=float(rep.candidate.mean()); ctrl=float(rep.control.mean()); delta=cm-ctrl; p10=float(np.quantile(rep.candidate,.10))
ax=oof.groupby(['repeat','axis']).agg(candidate=('candidate_correct','mean'),control=('control_correct','mean')).reset_index().groupby('axis').agg(candidate=('candidate','mean'),control=('control','mean')).reset_index(); ax['delta']=ax.candidate-ax.control; floor=float(ax.candidate.min())
ctrl_once=float(correct(pdff.control_diff).mean()); ties=int((pdff.control_diff.abs()<=1e-12).sum()); assert abs(ctrl_once-0.5544554455445545)<1e-9,(ctrl_once,ties)
rep.to_csv(ART/'V4_CONTEXTUAL_SEMANTIC_REPEAT_SUMMARY.csv',index=False); ax.to_csv(ART/'V4_CONTEXTUAL_SEMANTIC_AXIS_DIAGNOSTICS.csv',index=False); pd.DataFrame([{'official_control_accuracy':ctrl_once,'control_ties':ties,'tie_value':.5,'age_younger_fixed':float(pdff.positive_earlier_calc.mean())}]).to_csv(ART/'V4_CONTEXTUAL_SEMANTIC_BENCHMARK_AUDIT.csv',index=False)
display(rep); display(ax); print('candidate',cm,'control',ctrl,'delta',delta,'p10',p10,'axis floor',floor,'ties',ties)


repeat 1 /10
repeat 2 /10
repeat 3 /10
repeat 4 /10
repeat 5 /10
repeat 6 /10
repeat 7 /10
repeat 8 /10
repeat 9 /10
repeat 10 /10


,repeat,candidate,control,delta
0,0,0.455446,0.554455,-0.099010
1,1,0.534653,0.554455,-0.019802
2,2,0.495050,0.554455,-0.059406
3,3,0.514851,0.554455,-0.039604
4,4,0.524752,0.554455,-0.029703
5,5,0.524752,0.554455,-0.029703
6,6,0.475248,0.554455,-0.079208
7,7,0.534653,0.554455,-0.019802
8,8,0.574257,0.554455,0.019802
9,9,0.495050,0.554455,-0.059406


,axis,candidate,control,delta
0,COMPETITIVE,0.538710,0.548387,-0.009677
1,PROJECT,0.464000,0.620000,-0.156000
2,STATUS,0.522222,0.522222,0.000000


candidate 0.5128712871287128 control 0.5544554455445546 delta -0.04158415841584184 p10 0.47326732673267324 axis floor 0.46399999999999997 ties 6


In [8]:

# Subject bootstrap and final stopping gate.
sa=oof.groupby('subject_id').agg(candidate=('candidate_correct','mean'),control=('control_correct','mean')); ids=sa.index.to_numpy(); rng=np.random.RandomState(SEED+777); bd=np.empty(N_BOOTSTRAP)
for i in range(N_BOOTSTRAP):
    s=rng.choice(ids,size=len(ids),replace=True); bd[i]=float((sa.loc[s,'candidate'].to_numpy()-sa.loc[s,'control'].to_numpy()).mean())
boot=pd.DataFrame([{'observed_delta_vs_control':float((sa.candidate-sa.control).mean()),'bootstrap_mean_delta':float(bd.mean()),'ci025':float(np.quantile(bd,.025)),'ci975':float(np.quantile(bd,.975)),'p_delta_gt_0':float((bd>0).mean())}]); boot.to_csv(ART/'V4_CONTEXTUAL_SEMANTIC_BOOTSTRAP.csv',index=False); display(boot); bp=float(boot.iloc[0].p_delta_gt_0)
gate={'overall_ge_056':cm>=MIN_OVERALL,'delta_vs_control_ge_001':delta>=MIN_DELTA_CONTROL,'bootstrap_P_gt_0_ge_070':bp>=MIN_BOOT_P,'p10_ge_050':p10>=MIN_P10,'axis_floor_ge_048':floor>=MIN_AXIS}; passed=all(gate.values()); status='V4_CONTEXTUAL_SEMANTIC_STACK_V1_FROZEN_READY_FOR_FRESH_CONFIRM_DEV' if passed else 'V4_CONTEXTUAL_SEMANTIC_STACK_V1_REJECTED_END_101_DEV_MODEL_SEARCH'; print(gate,status)


,observed_delta_vs_control,bootstrap_mean_delta,ci025,ci975,p_delta_gt_0
0,-0.041584,-0.041496,-0.155446,0.073267,0.23995


{'overall_ge_056': False, 'delta_vs_control_ge_001': False, 'bootstrap_P_gt_0_ge_070': False, 'p10_ge_050': False, 'axis_floor_ge_048': False} V4_CONTEXTUAL_SEMANTIC_STACK_V1_REJECTED_END_101_DEV_MODEL_SEARCH


In [9]:

# Conditional freeze + decision/lineage.
model_path=ART/'V4_CONTEXTUAL_SEMANTIC_STACK_V1_FROZEN_MODEL.pkl'; weights_path=ART/'V4_CONTEXTUAL_SEMANTIC_STACK_V1_FROZEN_WEIGHTS.json'; coef_path=ART/'V4_CONTEXTUAL_SEMANTIC_STACK_V1_COEFFICIENTS.csv'
if passed:
    fm=fit(pdff); pickle.dump({'architecture':'CONTEXTUAL_SEMANTIC_STACK_V1','model':fm,'features':use,'C':MODEL_C},open(model_path,'wb')); sc=fm.named_steps['scale']; lr=fm.named_steps['model']; json.dump({'architecture':'CONTEXTUAL_SEMANTIC_STACK_V1','features':use,'scaler_mean':[float(x) for x in sc.mean_],'scaler_scale':[float(x) for x in sc.scale_],'coefficient':[float(x) for x in lr.coef_.ravel()],'intercept':float(lr.intercept_[0]),'C':MODEL_C},open(weights_path,'w',encoding='utf-8'),ensure_ascii=False,indent=2); pd.DataFrame({'feature':use,'coefficient':lr.coef_.ravel()}).assign(abs_coefficient=lambda x:x.coefficient.abs()).sort_values('abs_coefficient',ascending=False).to_csv(coef_path,index=False)
lineage={'created_at':datetime.now().isoformat(timespec='seconds'),'notebook_version':NOTEBOOK_VERSION,'spec_sha256':sha(SPEC),'pair_sha256':sha(PAIR_PATH),'w1_roster_sha256':sha(W1),'w2_roster_sha256':sha(W2),'saju_engine_sha256':sha(ROOT/'saju_engine.py'),'orthodox_helper_sha256':sha(ORTHO),'frozen_model_sha256':sha(model_path) if model_path.exists() else None,'frozen_weights_sha256':sha(weights_path) if weights_path.exists() else None}; json.dump(lineage,open(ART/'V4_CONTEXTUAL_SEMANTIC_STACK_V1_LINEAGE.json','w',encoding='utf-8'),ensure_ascii=False,indent=2)
decision={'version':'V4_CONTEXTUAL_SEMANTIC_STACK_V1_DECISION','notebook_version':NOTEBOOK_VERSION,'created_at':datetime.now().isoformat(timespec='seconds'),'status':status,'architecture':'CONTEXTUAL_SEMANTIC_STACK_V1','corpus_role':'CONSUMED_FINAL_ARCHITECTURE_DEVELOPMENT','n_pairs':101,'candidate_is_control_independent':True,'block_feats_called':False,'metrics':{'candidate_outer_cv_mean':cm,'control_fixed_mean':ctrl,'official_control_once':ctrl_once,'control_ties_scored_half':ties,'delta_vs_control':delta,'candidate_outer_cv_p10':p10,'candidate_axis_floor':floor,'bootstrap_P_delta_gt_0_vs_control':bp},'gate':gate,'dev_sanity_pass':bool(passed),'stopping_rule':'PASS: freeze and fresh confirmation DEV. FAIL: architecture/model search on these 101 pairs is permanently closed; next work must be larger/better fresh DEV and ground-truth redesign.','holdout_integrity':{'NEW_CONFIRM_loaded':False,'Validation_B_loaded':False,'Public_CHECK_loaded':False,'Public_FINAL_loaded':False}}; json.dump(decision,open(ART/'V4_CONTEXTUAL_SEMANTIC_STACK_V1_DECISION.json','w',encoding='utf-8'),ensure_ascii=False,indent=2); print(json.dumps(decision,ensure_ascii=False,indent=2))


{
  "version": "V4_CONTEXTUAL_SEMANTIC_STACK_V1_DECISION",
  "notebook_version": "SAJU_ML_V4_CONTEXTUAL_SEMANTIC_STACK_V1_20260817",
  "created_at": "2026-08-17T01:58:08",
  "status": "V4_CONTEXTUAL_SEMANTIC_STACK_V1_REJECTED_END_101_DEV_MODEL_SEARCH",
  "architecture": "CONTEXTUAL_SEMANTIC_STACK_V1",
  "corpus_role": "CONSUMED_FINAL_ARCHITECTURE_DEVELOPMENT",
  "n_pairs": 101,
  "candidate_is_control_independent": true,
  "block_feats_called": false,
  "metrics": {
    "candidate_outer_cv_mean": 0.5128712871287128,
    "control_fixed_mean": 0.5544554455445546,
    "official_control_once": 0.5544554455445545,
    "control_ties_scored_half": 6,
    "delta_vs_control": -0.04158415841584184,
    "candidate_outer_cv_p10": 0.47326732673267324,
    "candidate_axis_floor": 0.46399999999999997,
    "bootstrap_P_delta_gt_0_vs_control": 0.23995
  },
  "gate": {
    "overall_ge_056": false,
    "delta_vs_control_ge_001": false,
    "bootstrap_P_gt_0_ge_070": false,
    "p10_ge_050": false,
    "a

## Send back

Always send:

```text
V4_CONTEXTUAL_SEMANTIC_STACK_V1_DECISION.json
V4_CONTEXTUAL_SEMANTIC_REPEAT_SUMMARY.csv
V4_CONTEXTUAL_SEMANTIC_AXIS_DIAGNOSTICS.csv
V4_CONTEXTUAL_SEMANTIC_BOOTSTRAP.csv
V4_CONTEXTUAL_SEMANTIC_BENCHMARK_AUDIT.csv
V4_CONTEXTUAL_SEMANTIC_FEATURE_SCHEMA.json
V4_CONTEXTUAL_SEMANTIC_STACK_V1_LINEAGE.json
V4_CONTEXTUAL_HELPER_LEAKAGE_AUDIT.csv
```

Only if PASS/frozen, also send:

```text
V4_CONTEXTUAL_SEMANTIC_STACK_V1_COEFFICIENTS.csv
V4_CONTEXTUAL_SEMANTIC_STACK_V1_FROZEN_WEIGHTS.json
```